# Support Vector Machines (SVM) - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles, make_blobs, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is a Support Vector Machine?

Support Vector Machine (SVM) is a powerful supervised learning algorithm used for **classification** and **regression** tasks. The goal of SVM is to find the optimal hyperplane that separates data points of different classes with the **maximum margin**.

### Maximum Margin Hyperplane

The hyperplane is defined as:
$$\mathbf{w}^T\mathbf{x} + b = 0$$

where:
- $\mathbf{w}$ is the weight vector (normal to the hyperplane)
- $\mathbf{x}$ is the input feature vector
- $b$ is the bias term

The **margin** is the distance between the hyperplane and the nearest data points from each class. SVM finds the hyperplane that maximizes this margin.

The margin width is:
$$\text{margin} = \frac{2}{||\mathbf{w}||}$$

### Support Vectors

**Support vectors** are the data points that lie closest to the decision boundary. They are the critical elements that define the hyperplane:
- They satisfy: $y_i(\mathbf{w}^T\mathbf{x}_i + b) = 1$
- Removing non-support vector points doesn't change the decision boundary
- The number of support vectors indicates model complexity

### Hard Margin vs Soft Margin (C Parameter)

**Hard Margin SVM**: Requires perfect linear separation (no misclassifications allowed)
$$\min_{\mathbf{w}, b} \frac{1}{2}||\mathbf{w}||^2$$
$$\text{subject to: } y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1, \forall i$$

**Soft Margin SVM**: Allows some misclassifications via slack variables $\xi_i$
$$\min_{\mathbf{w}, b, \xi} \frac{1}{2}||\mathbf{w}||^2 + C\sum_{i=1}^{n}\xi_i$$
$$\text{subject to: } y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

The **C parameter** controls the trade-off:
- **Large C**: Less regularization, narrow margin, fits training data closely (may overfit)
- **Small C**: More regularization, wider margin, allows more margin violations (may underfit)

### The Kernel Trick

The kernel trick allows SVM to find non-linear decision boundaries by implicitly mapping data to a higher-dimensional feature space.

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^T \phi(\mathbf{x}_j)$$

Common kernels:

1. **Linear Kernel**:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i^T \mathbf{x}_j$$

2. **Polynomial Kernel**:
$$K(\mathbf{x}_i, \mathbf{x}_j) = (\gamma \mathbf{x}_i^T \mathbf{x}_j + r)^d$$

3. **Radial Basis Function (RBF/Gaussian) Kernel**:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma ||\mathbf{x}_i - \mathbf{x}_j||^2)$$

### Hinge Loss

For gradient-based optimization, SVM uses the **hinge loss**:
$$L(y, f(\mathbf{x})) = \max(0, 1 - y \cdot f(\mathbf{x}))$$

The full objective function becomes:
$$J(\mathbf{w}) = \frac{\lambda}{2}||\mathbf{w}||^2 + \frac{1}{n}\sum_{i=1}^{n}\max(0, 1 - y_i(\mathbf{w}^T\mathbf{x}_i + b))$$

where $\lambda = \frac{1}{C}$ is the regularization parameter.

### Time & Space Complexity

| Aspect | Linear SVM | Kernel SVM |
|--------|------------|------------|
| Training Time | $O(n \cdot d)$ to $O(n^2 \cdot d)$ | $O(n^2)$ to $O(n^3)$ |
| Prediction Time | $O(d)$ | $O(n_{sv} \cdot d)$ |
| Space | $O(d)$ | $O(n_{sv} \cdot d)$ |

where:
- $n$ = number of training samples
- $d$ = number of features
- $n_{sv}$ = number of support vectors

## 2. Implementation from Scratch <a id='implementation'></a>

We'll implement a Linear SVM using **gradient descent** on the hinge loss. This is for educational purposes - production implementations use more sophisticated optimization (e.g., SMO algorithm).

### Mathematical Derivation

**Objective Function:**
$$J(\mathbf{w}, b) = \frac{\lambda}{2}||\mathbf{w}||^2 + \frac{1}{n}\sum_{i=1}^{n}\max(0, 1 - y_i(\mathbf{w}^T\mathbf{x}_i + b))$$

**Gradients:**

For each sample $(\mathbf{x}_i, y_i)$:

If $y_i(\mathbf{w}^T\mathbf{x}_i + b) < 1$ (misclassified or within margin):
$$\frac{\partial J}{\partial \mathbf{w}} = \lambda\mathbf{w} - y_i\mathbf{x}_i$$
$$\frac{\partial J}{\partial b} = -y_i$$

If $y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1$ (correctly classified outside margin):
$$\frac{\partial J}{\partial \mathbf{w}} = \lambda\mathbf{w}$$
$$\frac{\partial J}{\partial b} = 0$$

In [ ]:
class LinearSVMScratch:
    """
    Linear Support Vector Machine implementation from scratch using gradient descent.
    
    This implementation uses the hinge loss and gradient descent for optimization.
    For educational purposes - not optimized for large-scale problems.
    
    Parameters:
    -----------
    learning_rate : float, default=0.001
        Learning rate for gradient descent
    lambda_param : float, default=0.01
        Regularization parameter (inverse of C)
    n_iterations : int, default=1000
        Number of iterations for gradient descent
    verbose : bool, default=False
        Print loss during training
    """
    
    def __init__(self, learning_rate=0.001, lambda_param=0.01, 
                 n_iterations=1000, verbose=False):
        self.learning_rate = learning_rate
        self.lambda_param = lambda_param  # Regularization (1/C)
        self.n_iterations = n_iterations
        self.verbose = verbose
        self.weights = None
        self.bias = None
        self.losses = []
        self.support_vectors_ = None
        self.support_vector_labels_ = None
        
    def _initialize_parameters(self, n_features):
        """
        Initialize weights using small random values.
        """
        # Small random initialization for stability
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0
        
    def _compute_hinge_loss(self, X, y):
        """
        Compute the hinge loss with L2 regularization.
        
        Loss = (lambda/2) * ||w||^2 + (1/n) * sum(max(0, 1 - y_i * (w^T x_i + b)))
        """
        n_samples = X.shape[0]
        
        # Compute margins: y_i * (w^T x_i + b)
        margins = y * (np.dot(X, self.weights) + self.bias)
        
        # Hinge loss: max(0, 1 - margin)
        hinge_loss = np.maximum(0, 1 - margins)
        
        # Total loss with regularization
        regularization = (self.lambda_param / 2) * np.dot(self.weights, self.weights)
        total_loss = regularization + np.mean(hinge_loss)
        
        return total_loss
    
    def _compute_gradients(self, X, y):
        """
        Compute gradients for weights and bias.
        
        For each sample:
        - If y_i * (w^T x_i + b) < 1: dw = lambda*w - y_i*x_i, db = -y_i
        - Otherwise: dw = lambda*w, db = 0
        """
        n_samples = X.shape[0]
        
        # Compute margins
        margins = y * (np.dot(X, self.weights) + self.bias)
        
        # Initialize gradients with regularization term
        dw = self.lambda_param * self.weights
        db = 0.0
        
        # Add hinge loss gradients for misclassified/margin samples
        for i in range(n_samples):
            if margins[i] < 1:
                dw -= y[i] * X[i] / n_samples
                db -= y[i] / n_samples
                
        return dw, db
    
    def _compute_gradients_vectorized(self, X, y):
        """
        Vectorized gradient computation (faster for large datasets).
        """
        n_samples = X.shape[0]
        
        # Compute margins
        margins = y * (np.dot(X, self.weights) + self.bias)
        
        # Mask for samples within margin or misclassified
        mask = margins < 1
        
        # Compute gradients
        dw = self.lambda_param * self.weights - np.dot(X.T, y * mask) / n_samples
        db = -np.sum(y * mask) / n_samples
        
        return dw, db
    
    def fit(self, X, y):
        """
        Train the SVM model using gradient descent.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target labels (-1 or 1)
        """
        # Convert to numpy arrays
        X = np.array(X)
        y = np.array(y)
        
        # Convert labels to -1, 1 if they are 0, 1
        if set(np.unique(y)) == {0, 1}:
            y = 2 * y - 1  # Convert 0,1 to -1,1
        
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self._initialize_parameters(n_features)
        
        # Gradient descent
        for iteration in range(self.n_iterations):
            # Compute loss
            loss = self._compute_hinge_loss(X, y)
            self.losses.append(loss)
            
            # Compute gradients (using vectorized version)
            dw, db = self._compute_gradients_vectorized(X, y)
            
            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Print progress
            if self.verbose and iteration % 100 == 0:
                print(f'Iteration {iteration}, Loss: {loss:.4f}')
        
        # Identify support vectors (points within margin)
        margins = y * (np.dot(X, self.weights) + self.bias)
        sv_mask = margins <= 1 + 1e-7  # Points on or within margin
        self.support_vectors_ = X[sv_mask]
        self.support_vector_labels_ = y[sv_mask]
        
        return self
    
    def decision_function(self, X):
        """
        Compute the decision function value (signed distance to hyperplane).
        """
        X = np.array(X)
        return np.dot(X, self.weights) + self.bias
    
    def predict(self, X):
        """
        Predict class labels.
        
        Returns:
        --------
        predictions : array, values in {-1, 1}
        """
        decision = self.decision_function(X)
        return np.sign(decision)
    
    def predict_01(self, X):
        """
        Predict class labels in {0, 1} format.
        """
        predictions = self.predict(X)
        return ((predictions + 1) / 2).astype(int)
    
    def score(self, X, y):
        """
        Return accuracy score.
        """
        y = np.array(y)
        if set(np.unique(y)) == {0, 1}:
            y = 2 * y - 1
        predictions = self.predict(X)
        return np.mean(predictions == y)
    
    def get_margin_width(self):
        """
        Compute the margin width: 2 / ||w||
        """
        return 2 / np.linalg.norm(self.weights)

In [ ]:
# Test our implementation on a simple linearly separable dataset
print("Testing Linear SVM Implementation")
print("=" * 50)

# Create linearly separable data
X_simple, y_simple = make_blobs(n_samples=200, centers=2, n_features=2, 
                                 cluster_std=1.5, random_state=42)

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X_simple, y_simple, 
                                                     test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train our SVM
svm_scratch = LinearSVMScratch(learning_rate=0.01, lambda_param=0.01, 
                                n_iterations=1000, verbose=True)
svm_scratch.fit(X_train_scaled, y_train)

print(f"\nTraining Accuracy: {svm_scratch.score(X_train_scaled, y_train):.4f}")
print(f"Test Accuracy: {svm_scratch.score(X_test_scaled, y_test):.4f}")
print(f"Number of Support Vectors: {len(svm_scratch.support_vectors_)}")
print(f"Margin Width: {svm_scratch.get_margin_width():.4f}")

## 3. Training & Optimization <a id='training'></a>

Now let's explore SVM with **non-linear** datasets using sklearn's SVC with different kernels.

In [ ]:
# Create non-linear datasets
# Make moons - crescent shaped data
X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=42)

# Make circles - concentric circles
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)

# Visualize the datasets
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Moons dataset
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='RdYlBu', 
                edgecolor='black', s=50)
axes[0].set_title('Make Moons Dataset', fontsize=12)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Circles dataset
axes[1].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='RdYlBu', 
                edgecolor='black', s=50)
axes[1].set_title('Make Circles Dataset', fontsize=12)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("Note: These datasets are NOT linearly separable!")
print("A linear SVM will perform poorly. We need kernel methods.")

In [ ]:
def train_and_evaluate_svm(X, y, kernel='rbf', C=1.0, gamma='scale', degree=3):
    """
    Train SVM with given parameters and return model with metrics.
    """
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                         random_state=42)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train SVM
    svm = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree, random_state=42)
    svm.fit(X_train_scaled, y_train)
    
    # Evaluate
    train_acc = svm.score(X_train_scaled, y_train)
    test_acc = svm.score(X_test_scaled, y_test)
    n_sv = svm.n_support_.sum()
    
    return {
        'model': svm,
        'scaler': scaler,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'n_support_vectors': n_sv,
        'X_train': X_train_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train,
        'y_test': y_test
    }

# Test different kernels on moons dataset
print("Training SVMs on Make Moons Dataset")
print("=" * 60)

kernels = ['linear', 'poly', 'rbf']
moons_results = {}

for kernel in kernels:
    result = train_and_evaluate_svm(X_moons, y_moons, kernel=kernel, C=1.0)
    moons_results[kernel] = result
    print(f"\n{kernel.upper()} Kernel:")
    print(f"  Train Accuracy: {result['train_acc']:.4f}")
    print(f"  Test Accuracy: {result['test_acc']:.4f}")
    print(f"  Support Vectors: {result['n_support_vectors']}")

In [ ]:
# Test different kernels on circles dataset
print("Training SVMs on Make Circles Dataset")
print("=" * 60)

circles_results = {}

for kernel in kernels:
    result = train_and_evaluate_svm(X_circles, y_circles, kernel=kernel, C=1.0)
    circles_results[kernel] = result
    print(f"\n{kernel.upper()} Kernel:")
    print(f"  Train Accuracy: {result['train_acc']:.4f}")
    print(f"  Test Accuracy: {result['test_acc']:.4f}")
    print(f"  Support Vectors: {result['n_support_vectors']}")

In [ ]:
# Effect of C parameter
print("\nEffect of C Parameter (Regularization)")
print("=" * 60)

C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
c_results = []

for C in C_values:
    result = train_and_evaluate_svm(X_moons, y_moons, kernel='rbf', C=C)
    c_results.append({
        'C': C,
        'train_acc': result['train_acc'],
        'test_acc': result['test_acc'],
        'n_sv': result['n_support_vectors']
    })
    print(f"C={C:6.2f}: Train={result['train_acc']:.4f}, "
          f"Test={result['test_acc']:.4f}, SVs={result['n_support_vectors']}")

print("\nObservation:")
print("- Small C: More regularization, wider margin, more SVs")
print("- Large C: Less regularization, narrower margin, fewer SVs")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
def plot_confusion_matrices(results_dict, dataset_name):
    """
    Plot confusion matrices for different kernels.
    """
    fig, axes = plt.subplots(1, len(results_dict), figsize=(15, 4))
    
    for idx, (kernel, result) in enumerate(results_dict.items()):
        y_pred = result['model'].predict(result['X_test'])
        cm = confusion_matrix(result['y_test'], y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                   xticklabels=['Class 0', 'Class 1'],
                   yticklabels=['Class 0', 'Class 1'])
        axes[idx].set_title(f'{kernel.upper()} Kernel\nAccuracy: {result["test_acc"]:.3f}')
        axes[idx].set_ylabel('True Label')
        axes[idx].set_xlabel('Predicted Label')
    
    plt.suptitle(f'Confusion Matrices - {dataset_name}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Plot confusion matrices
plot_confusion_matrices(moons_results, 'Make Moons Dataset')
plot_confusion_matrices(circles_results, 'Make Circles Dataset')

In [ ]:
def visualize_support_vectors(X, y, model, scaler, title='Support Vector Visualization'):
    """
    Visualize the support vectors and decision boundary.
    """
    # Scale data for visualization
    X_scaled = scaler.transform(X)
    
    # Create mesh for decision boundary
    h = 0.02
    x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
    y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Get decision function values
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(10, 8))
    
    # Decision boundary and margins
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu', levels=50)
    plt.contour(xx, yy, Z, colors='black', levels=[-1, 0, 1], 
                linestyles=['--', '-', '--'], linewidths=[1, 2, 1])
    
    # Plot all points
    plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdYlBu', 
                edgecolor='black', s=50, label='Data points')
    
    # Highlight support vectors
    sv = model.support_vectors_
    plt.scatter(sv[:, 0], sv[:, 1], s=200, facecolors='none', 
                edgecolors='green', linewidths=2, label='Support Vectors')
    
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(f'{title}\nSupport Vectors: {len(sv)}')
    plt.legend(loc='upper right')
    plt.colorbar(label='Decision Function Value')
    plt.show()

# Visualize support vectors for RBF kernel on moons
result = moons_results['rbf']
visualize_support_vectors(X_moons, y_moons, result['model'], result['scaler'],
                          'RBF Kernel - Moons Dataset')

In [ ]:
# Detailed classification report
print("Detailed Classification Report - RBF Kernel on Moons Dataset")
print("=" * 60)

result = moons_results['rbf']
y_pred = result['model'].predict(result['X_test'])
print(classification_report(result['y_test'], y_pred, 
                           target_names=['Class 0', 'Class 1']))

print("\nDetailed Classification Report - RBF Kernel on Circles Dataset")
print("=" * 60)

result = circles_results['rbf']
y_pred = result['model'].predict(result['X_test'])
print(classification_report(result['y_test'], y_pred, 
                           target_names=['Class 0', 'Class 1']))

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
def plot_decision_boundaries_comparison(X, y, dataset_name):
    """
    Plot decision boundaries for different kernels side by side.
    """
    # Prepare data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Define kernels and parameters
    configs = [
        ('Linear', {'kernel': 'linear', 'C': 1.0}),
        ('Polynomial (d=3)', {'kernel': 'poly', 'C': 1.0, 'degree': 3}),
        ('RBF', {'kernel': 'rbf', 'C': 1.0, 'gamma': 'scale'}),
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Create mesh
    h = 0.02
    x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
    y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    for idx, (name, params) in enumerate(configs):
        # Train model
        svm = SVC(**params, random_state=42)
        svm.fit(X_scaled, y)
        
        # Get predictions
        Z = svm.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        
        # Plot
        axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
        axes[idx].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdYlBu', 
                         edgecolor='black', s=30)
        
        # Highlight support vectors
        axes[idx].scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
                         s=100, facecolors='none', edgecolors='green', linewidths=2)
        
        acc = svm.score(X_scaled, y)
        axes[idx].set_title(f'{name}\nAccuracy: {acc:.3f}, SVs: {len(svm.support_vectors_)}')
        axes[idx].set_xlabel('Feature 1')
        axes[idx].set_ylabel('Feature 2')
    
    plt.suptitle(f'Decision Boundaries Comparison - {dataset_name}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Plot for both datasets
plot_decision_boundaries_comparison(X_moons, y_moons, 'Make Moons')
plot_decision_boundaries_comparison(X_circles, y_circles, 'Make Circles')

In [ ]:
def plot_margin_visualization(X, y):
    """
    Visualize the margin for a linear SVM.
    """
    # Use linearly separable data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Train linear SVM
    svm = SVC(kernel='linear', C=1.0, random_state=42)
    svm.fit(X_scaled, y)
    
    # Get model parameters
    w = svm.coef_[0]
    b = svm.intercept_[0]
    
    # Create mesh
    x_min, x_max = X_scaled[:, 0].min() - 1, X_scaled[:, 0].max() + 1
    y_min, y_max = X_scaled[:, 1].min() - 1, X_scaled[:, 1].max() + 1
    
    # Plot
    plt.figure(figsize=(10, 8))
    
    # Plot data points
    plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdYlBu', 
                edgecolor='black', s=80, zorder=3)
    
    # Highlight support vectors
    plt.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
                s=200, facecolors='none', edgecolors='green', linewidths=3, 
                label='Support Vectors', zorder=4)
    
    # Decision boundary: w[0]*x + w[1]*y + b = 0
    # => y = -(w[0]*x + b) / w[1]
    x_line = np.linspace(x_min, x_max, 100)
    
    # Decision boundary (w.x + b = 0)
    y_decision = -(w[0] * x_line + b) / w[1]
    
    # Margin boundaries (w.x + b = +/- 1)
    y_margin_pos = -(w[0] * x_line + b + 1) / w[1]  # w.x + b = -1
    y_margin_neg = -(w[0] * x_line + b - 1) / w[1]  # w.x + b = +1
    
    # Plot lines
    plt.plot(x_line, y_decision, 'k-', linewidth=2, label='Decision Boundary')
    plt.plot(x_line, y_margin_pos, 'k--', linewidth=1, label='Margin')
    plt.plot(x_line, y_margin_neg, 'k--', linewidth=1)
    
    # Fill margin region
    plt.fill_between(x_line, y_margin_pos, y_margin_neg, alpha=0.2, color='yellow',
                    label='Margin Region')
    
    # Calculate margin width
    margin_width = 2 / np.linalg.norm(w)
    
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.xlabel('Feature 1', fontsize=12)
    plt.ylabel('Feature 2', fontsize=12)
    plt.title(f'Linear SVM - Margin Visualization\nMargin Width: {margin_width:.3f}', fontsize=14)
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return svm

# Create linearly separable data for margin visualization
X_linear, y_linear = make_blobs(n_samples=100, centers=2, n_features=2, 
                                 cluster_std=1.0, random_state=42)
svm_linear = plot_margin_visualization(X_linear, y_linear)

In [ ]:
def plot_c_parameter_effect(X, y):
    """
    Visualize the effect of C parameter on decision boundary.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    # Create mesh
    h = 0.02
    x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
    y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    for idx, C in enumerate(C_values):
        # Train SVM
        svm = SVC(kernel='rbf', C=C, gamma='scale', random_state=42)
        svm.fit(X_scaled, y)
        
        # Get predictions
        Z = svm.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        
        # Plot
        axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
        axes[idx].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdYlBu', 
                         edgecolor='black', s=20)
        axes[idx].scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
                         s=60, facecolors='none', edgecolors='green', linewidths=1.5)
        
        acc = svm.score(X_scaled, y)
        axes[idx].set_title(f'C = {C}\nAcc: {acc:.2f}, SVs: {len(svm.support_vectors_)}')
        axes[idx].set_xlabel('Feature 1')
        if idx == 0:
            axes[idx].set_ylabel('Feature 2')
    
    plt.suptitle('Effect of C Parameter (RBF Kernel) - Small C = Wide Margin, Large C = Narrow Margin', 
                 fontsize=12, y=1.05)
    plt.tight_layout()
    plt.show()

# Show C parameter effect on moons dataset
plot_c_parameter_effect(X_moons, y_moons)

In [ ]:
def plot_gamma_effect(X, y):
    """
    Visualize the effect of gamma parameter on RBF kernel.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    gamma_values = [0.01, 0.1, 1.0, 10.0, 100.0]
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    # Create mesh
    h = 0.02
    x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
    y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    for idx, gamma in enumerate(gamma_values):
        # Train SVM
        svm = SVC(kernel='rbf', C=1.0, gamma=gamma, random_state=42)
        svm.fit(X_scaled, y)
        
        # Get predictions
        Z = svm.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        
        # Plot
        axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
        axes[idx].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdYlBu', 
                         edgecolor='black', s=20)
        axes[idx].scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
                         s=60, facecolors='none', edgecolors='green', linewidths=1.5)
        
        acc = svm.score(X_scaled, y)
        axes[idx].set_title(f'gamma = {gamma}\nAcc: {acc:.2f}, SVs: {len(svm.support_vectors_)}')
        axes[idx].set_xlabel('Feature 1')
        if idx == 0:
            axes[idx].set_ylabel('Feature 2')
    
    plt.suptitle('Effect of Gamma Parameter (RBF Kernel) - Small = Smooth, Large = Complex', 
                 fontsize=12, y=1.05)
    plt.tight_layout()
    plt.show()

# Show gamma effect on moons dataset
plot_gamma_effect(X_moons, y_moons)

In [ ]:
# Plot training loss curve from our scratch implementation
plt.figure(figsize=(10, 5))
plt.plot(svm_scratch.losses, linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Hinge Loss + Regularization', fontsize=12)
plt.title('Training Loss Curve - Linear SVM (Scratch Implementation)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use SVM

#### Good Use Cases:

1. **High-Dimensional Data**
   - Text classification (document categorization, sentiment analysis)
   - Gene expression data analysis
   - Image classification with extracted features
   - Works well when n_features >> n_samples

2. **Clear Margin Separation**
   - Data with distinct class boundaries
   - Problems where classes don't heavily overlap

3. **Small to Medium Datasets**
   - Training set < 10,000-100,000 samples
   - Memory-efficient for predictions

4. **Binary Classification**
   - Spam detection
   - Fraud detection
   - Medical diagnosis (disease vs. healthy)

5. **Non-linear Problems**
   - Use kernel trick (RBF, polynomial) for complex boundaries

#### When NOT to Use SVM:

1. **Very Large Datasets (>100,000 samples)**
   - Training time scales poorly: O(n^2) to O(n^3)
   - Consider: SGDClassifier, Random Forest, or Neural Networks

2. **Many Features with Noisy Data**
   - SVM can overfit with too many irrelevant features
   - Consider: Feature selection first, or tree-based methods

3. **Multi-class Problems with Many Classes**
   - SVM is inherently binary; multi-class requires one-vs-one or one-vs-rest
   - Consider: Neural Networks, Random Forest for many classes

4. **Need for Probability Estimates**
   - SVM doesn't naturally produce probabilities
   - Platt scaling can be used but adds computation
   - Consider: Logistic Regression, Neural Networks

5. **Highly Imbalanced Data**
   - SVM can be biased toward majority class
   - Use class_weight='balanced' or resampling techniques

### Pros and Cons

| Pros | Cons |
|------|------|
| Effective in high dimensions | Slow on large datasets |
| Memory efficient (uses support vectors) | Sensitive to feature scaling |
| Versatile kernel functions | No probability output by default |
| Robust to overfitting (with good C) | Choice of kernel can be tricky |
| Works well with clear margin | Doesn't handle noisy data well |

### Kernel Selection Guide

| Kernel | When to Use | Key Parameters |
|--------|-------------|----------------|
| **Linear** | Linearly separable data, high-dimensional text data | C |
| **RBF** | Default choice, non-linear data, unknown structure | C, gamma |
| **Polynomial** | Known polynomial relationship, image processing | C, degree, coef0 |
| **Sigmoid** | Neural network-like behavior (rarely used) | C, gamma, coef0 |

### Hyperparameter Tuning Guidelines

| Parameter | Range | Effect |
|-----------|-------|--------|
| C | 0.001 - 1000 | Higher = less regularization, tighter fit |
| gamma (RBF) | 0.001 - 100 | Higher = more complex boundary, risk of overfitting |
| degree (Poly) | 2 - 5 | Higher = more flexible, more computation |

### Best Practices

1. **Always scale features** (StandardScaler or MinMaxScaler)
2. **Start with RBF kernel** if unsure
3. **Use cross-validation** for hyperparameter tuning
4. **Try linear kernel first** for high-dimensional data
5. **Monitor number of support vectors** (fewer = simpler model)

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare our scratch implementation with sklearn's LinearSVC
from sklearn.svm import LinearSVC

print("Comparison: Scratch Implementation vs sklearn")
print("=" * 60)

# Use the same linearly separable dataset
X_compare, y_compare = make_blobs(n_samples=300, centers=2, n_features=2, 
                                   cluster_std=1.5, random_state=42)

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X_compare, y_compare, 
                                                     test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert labels to -1, 1 for our implementation
y_train_svm = 2 * y_train - 1
y_test_svm = 2 * y_test - 1

# Train our implementation
our_svm = LinearSVMScratch(learning_rate=0.01, lambda_param=0.01, n_iterations=2000)
our_svm.fit(X_train_scaled, y_train_svm)

# Train sklearn's SVC with linear kernel
sklearn_svc = SVC(kernel='linear', C=100.0, random_state=42)  # C = 1/lambda
sklearn_svc.fit(X_train_scaled, y_train)

# Train sklearn's LinearSVC
sklearn_lsvc = LinearSVC(C=100.0, max_iter=2000, random_state=42)
sklearn_lsvc.fit(X_train_scaled, y_train)

# Compare results
print("\nOur Implementation:")
print(f"  Train Accuracy: {our_svm.score(X_train_scaled, y_train_svm):.4f}")
print(f"  Test Accuracy: {our_svm.score(X_test_scaled, y_test_svm):.4f}")
print(f"  Weights: {our_svm.weights}")
print(f"  Bias: {our_svm.bias:.4f}")

print("\nsklearn SVC (linear kernel):")
print(f"  Train Accuracy: {sklearn_svc.score(X_train_scaled, y_train):.4f}")
print(f"  Test Accuracy: {sklearn_svc.score(X_test_scaled, y_test):.4f}")
print(f"  Weights: {sklearn_svc.coef_[0]}")
print(f"  Bias: {sklearn_svc.intercept_[0]:.4f}")

print("\nsklearn LinearSVC:")
print(f"  Train Accuracy: {sklearn_lsvc.score(X_train_scaled, y_train):.4f}")
print(f"  Test Accuracy: {sklearn_lsvc.score(X_test_scaled, y_test):.4f}")
print(f"  Weights: {sklearn_lsvc.coef_[0]}")
print(f"  Bias: {sklearn_lsvc.intercept_[0]:.4f}")

In [ ]:
# Visual comparison of decision boundaries
def plot_comparison_boundaries(X, y, our_model, sklearn_model, scaler):
    """
    Compare decision boundaries from scratch vs sklearn.
    """
    X_scaled = scaler.transform(X)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Create mesh
    h = 0.02
    x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
    y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Our implementation
    Z_our = our_model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z_our = Z_our.reshape(xx.shape)
    
    axes[0].contourf(xx, yy, Z_our, alpha=0.4, cmap='RdYlBu')
    axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdYlBu', 
                   edgecolor='black', s=50)
    if our_model.support_vectors_ is not None:
        axes[0].scatter(our_model.support_vectors_[:, 0], our_model.support_vectors_[:, 1],
                       s=150, facecolors='none', edgecolors='green', linewidths=2)
    axes[0].set_title(f'Our Implementation\nSVs: {len(our_model.support_vectors_) if our_model.support_vectors_ is not None else "N/A"}')
    axes[0].set_xlabel('Feature 1')
    axes[0].set_ylabel('Feature 2')
    
    # sklearn implementation
    Z_sklearn = sklearn_model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z_sklearn = Z_sklearn.reshape(xx.shape)
    
    axes[1].contourf(xx, yy, Z_sklearn, alpha=0.4, cmap='RdYlBu')
    axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdYlBu', 
                   edgecolor='black', s=50)
    axes[1].scatter(sklearn_model.support_vectors_[:, 0], sklearn_model.support_vectors_[:, 1],
                   s=150, facecolors='none', edgecolors='green', linewidths=2)
    axes[1].set_title(f'sklearn SVC\nSVs: {len(sklearn_model.support_vectors_)}')
    axes[1].set_xlabel('Feature 1')
    axes[1].set_ylabel('Feature 2')
    
    plt.suptitle('Decision Boundary Comparison: Scratch vs sklearn', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_comparison_boundaries(X_compare, y_compare, our_svm, sklearn_svc, scaler)

In [ ]:
# Comprehensive kernel comparison on both datasets
print("\nComprehensive Kernel Comparison")
print("=" * 70)

datasets = {
    'Moons': (X_moons, y_moons),
    'Circles': (X_circles, y_circles),
    'Blobs (Linear)': (X_linear, y_linear)
}

kernels = {
    'Linear': {'kernel': 'linear', 'C': 1.0},
    'Poly (d=2)': {'kernel': 'poly', 'degree': 2, 'C': 1.0},
    'Poly (d=3)': {'kernel': 'poly', 'degree': 3, 'C': 1.0},
    'RBF': {'kernel': 'rbf', 'C': 1.0, 'gamma': 'scale'},
}

results_table = []

for ds_name, (X, y) in datasets.items():
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)
    
    for k_name, k_params in kernels.items():
        svm = SVC(**k_params, random_state=42)
        svm.fit(X_tr_sc, y_tr)
        
        results_table.append({
            'Dataset': ds_name,
            'Kernel': k_name,
            'Train Acc': f"{svm.score(X_tr_sc, y_tr):.3f}",
            'Test Acc': f"{svm.score(X_te_sc, y_te):.3f}",
            'SVs': svm.n_support_.sum()
        })

# Print as formatted table
print(f"{'Dataset':<18} {'Kernel':<12} {'Train':<8} {'Test':<8} {'SVs':<6}")
print("-" * 55)
for r in results_table:
    print(f"{r['Dataset']:<18} {r['Kernel']:<12} {r['Train Acc']:<8} {r['Test Acc']:<8} {r['SVs']:<6}")

## Summary & Key Takeaways

### What We Learned:

1. **Mathematical Foundation**
   - SVM finds the maximum margin hyperplane
   - Support vectors are the critical points that define the boundary
   - Hinge loss enables gradient-based optimization

2. **Kernel Trick**
   - Linear kernel: Simple, fast, for linearly separable data
   - RBF kernel: Versatile, handles non-linear patterns
   - Polynomial kernel: For polynomial relationships

3. **Key Hyperparameters**
   - **C**: Regularization (larger = tighter fit, smaller = wider margin)
   - **gamma** (RBF): Influence of single training example (larger = more complex)

4. **Practical Considerations**
   - Always scale features before training
   - Start with RBF kernel for unknown data
   - Use linear kernel for high-dimensional data
   - Watch for overfitting with high C or gamma

### Key Insights:

- SVM is powerful for small-to-medium datasets with clear margins
- The kernel trick enables non-linear classification without explicit feature mapping
- Number of support vectors indicates model complexity
- Feature scaling is crucial for SVM performance

### When to Choose SVM:

- High-dimensional data (text, genomics)
- Clear class separation expected
- Medium-sized datasets
- Need for non-linear decision boundaries

### Next Steps:

- Implement kernelized SVM with SMO algorithm
- Explore multi-class SVM strategies
- Apply SVM to real-world datasets
- Compare with other classifiers (Random Forest, Neural Networks)